Bei anderen NB doppelte resample ünberprüfen dank prune

## Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import model_analysis

import importlib
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd

import numpy as np

import cartopy.crs as ccrs
from scipy.stats import beta



## Load Data

The code is capable of working with hist > 1 and r > 1, but if we use both it gets really slow, for obvious reasons

In [2]:
max_radius = 4
hist = 1
is_local_data = False
Month_idx = 4
safe = True
start_wanted = None  # later this will be shifted, if it is to close to the beginning of the data, such that there is allways data also for the hist dimension
end = None
max_depth = 3


In [3]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [4]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")


In [5]:

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
approx_maximas = max_v


In [6]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")
 

In [7]:
#min_start 

data_start = pd.to_datetime(raw_mrsol_for_mean.time[0].item())
offset = pd.tseries.frequencies.to_offset("ME")
min_start = data_start + pd.DateOffset(months=hist)

if start_wanted is None:
    start = min_start
else:
    start = pd.to_datetime(start_wanted) 
    start = max(min_start,start)
start = start.strftime("%Y-%m-%d")  

### some functions to shape the data

In [8]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = (ds/maximas)
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [9]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [10]:
def shape_input(ds, chunk_mask, hist, radius):
    ds = model.shape_data.add_hist_dimension(ds, hist)    
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.shape_data.add_radius_dimension(ds, radius)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

### Linear Regression of the mean

In [11]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [12]:
mean_predictor_sets  = {}
mean_max_predictor = shape_input(raw_input_for_mean,chunk_mask,hist, max_radius)
mean_max_predictor
for radius in range(0, max_radius):
   mean_predictor_sets[f"radius_{radius}"] = mean_max_predictor.sel(lon_translations=slice(-radius,radius)).sel(lat_translations=slice(-radius,radius))

In [13]:
Regr_set_mean = {}

In [14]:
for key,predictors in mean_predictor_sets.items():
    Regr_set_mean[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist","lon_translations","lat_translations"])
    Regr_set_mean[key].fit(predictors=predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

In [15]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [16]:
var_predictor_sets  = {}
var_max_predictor = shape_input(raw_input_for_var,chunk_mask,hist,max_radius)

for radius in range(0, max_radius):
    var_predictor_sets[f"radius_{radius}"] = var_max_predictor.sel(lon_translations=slice(-radius,radius)).sel(lat_translations=slice(-radius,radius))

In [17]:
residuals = {}
for key, regr in Regr_set_mean.items():
    residuals[key] = regr.residuals(var_predictor_sets[key], var_target,location_dim="gridcell", regr_dim="time")


### Linear Regression of the Variance

In [18]:
Regr_set_var = {}

In [19]:
for key, res in residuals.items():
    Regr_set_var[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist","lon_translations","lat_translations"])
    Regr_set_var[key].fit(predictors=var_predictor_sets[key], target=(res.residuals)**2,location_dim="gridcell", regr_dim="time")

### Export Prameters

In [20]:
if safe:
    for key, mean_regr in Regr_set_mean.items():
        model.save.save_params(mean_regr.params,Regr_set_var[key].params, maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/radius/{key}/local{is_local_data}/month{Month_idx}", name=f"hist={hist},start={start_wanted},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)
